In [ ]:
import numpy as np
import time
import os
from pynq import Overlay, allocate

: 

Load Bitstream Overlay

In [ ]:
print("[INFO] Loading bitstream onto FPGA...")
overlay = Overlay("design_1_wrapper.bit")
dma = overlay.axi_dma_0

: 

Define Dataset Parameters


In [ ]:
SEQUENCE_LEN = 128      # Length of each genomic sequence chunk
NUM_BYTES_PER_SAMPLE = 2 # 16-bit data width


Allocate Contiguous DMA Memory Buffers

In [ ]:
# Input buffer holds both reference and query sequences (2 * 128 = 256 samples of int16)
in_buffer = allocate(shape=(2 * SEQUENCE_LEN,), dtype=np.int16)
out_buffer = allocate(shape=(1,), dtype=np.int32)

def run_dtw_hardware(ref_seq, query_seq):
    """
    Sends two genomic sequences to the FPGA DTW engine via DMA
    and returns the calculated hardware DTW distance.
    """
    # Load data into physical DMA buffers
    in_buffer[:SEQUENCE_LEN] = ref_seq
    in_buffer[SEQUENCE_LEN:] = query_seq

    # Trigger AXI DMA transfers (Transmit & Receive concurrently)
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)

    # Wait for completion
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    return out_buffer[0]

Batch Processing Loop Across Dataset


In [ ]:
def process_dataset(dataset_folder):
    """
    Iterates through all numpy signal files in a folder, executes DTW 
    acceleration on hardware, and calculates total performance metrics.
    """
    file_list = [f for f in os.listdir(dataset_folder) if f.endswith('.npy')]
    results = []
    
    print(f"[INFO] Processing {len(file_list)} signal pairs...")
    
    start_time = time.perf_counter()
    
    for file_name in file_list:
        file_path = os.path.join(dataset_folder, file_name)
        data = np.load(file_path) # Assumes array containing ref and query
        
        ref_seq = data[0].astype(np.int16)
        query_seq = data[1].astype(np.int16)
        
        cost = run_dtw_hardware(ref_seq, query_seq)
        results.append((file_name, cost))
        
    end_time = time.perf_counter()
    
    total_time = end_time - start_time
    total_samples = len(file_list) * 2 * SEQUENCE_LEN
    throughput_msps = (total_samples / total_time) / 1e6
    
    print("=" * 45)
    print(f" Total Files Processed : {len(file_list)}")
    print(f" Execution Time        : {total_time:.4f} seconds")
    print(f" Hardware Throughput   : {throughput_msps:.2f} MSPS (Mega-samples/sec)")
    print("=" * 45)
    
    return results

if __name__ == "__main__":
    print("[SYSTEM READY] Run process_dataset('/path/to/dataset') to execute batch run.")